In [ ]:
### Learnt new

# .transform -> comme un OVER (PARTITION BY)
    df_reviews = pd.read_csv("../data/raw/reviews.csv", parse_dates=["date"])
    df["year_total"] = (
        df
        .groupby("year")["n_reviews"]
        .transform("sum")
    )

# .all() 
   assert np.isclose(table_to_check["pct_of_year"],100,atol=0.15).all() # -> The order is always: compare first, then reduce

# Percentiles 
    df["price_eur"].quantile([0.5, 0.9, 0.99])

# Save list of indexes: .index save indexes, .tolist() turns them into a list
    .index
    .tolist()

# Thousand separator 
    f"There are {len(df):,} lines"

### LINE CHART
    import matplotlib.pyplot as plt # -> the engine: 
    import seaborn as sns           # -> You give it a long DataFrame and column names, and it handles the grouping
    wide_df.plot() # Si wide dataset ? 

    fig, ax = plt.subplots(figsize=(10, 5))                                 # always start with fig, ax = plt.subplots().
    sns.lineplot(data=<long_df>, x="<col>", y="<col>", hue="<col>", ax=ax)  
        # hue -> le grouping 
        # marker="o"
    sns.despine() # remove clutter (les axes opposés)
        # sns.despine(bottom=True) -> les axes opposés + l'axe x
    ax.set_title("<the answer, as a sentence>")
        # loc="left", fontsize=10 -> controle the position and size
    fig.suptitle("<subtitle>")
        # x=0.01, ha="left", fontweight="bold") -> controle for position and type
    ax.set_xlabel("<label>")
    ax.set_xticks([]) # get rid of x-values 
    ax.set_ylabel("<label>")
    fig.text(0.01, -0.02, "Note: ...", fontsize=9)  # 0.01 et -0.02 : position dans le cadre du graphe
    fig.savefig("../docs/<name>.png", dpi=150, bbox_inches="tight")

fig.suptitle("Entire homes cost 2.3× more in Élysée than in Ménilmontant", x=0.01, ha="left", fontweight="bold")
ax.set_title("Median nightly price by neighbourhood; blue = above the Paris median", loc="left", fontsize=10)


### HISTOGRAM
    sns.histplot(data=df, x="price_eur", bins=50, ax=ax)

### BOX PLOT 
    sns.boxplot(data=df, x="room_type", y="price_eur", ax=ax, order=...)
    # -> order=.. should be a list of the category

### BAR CHART 
    sns.barplot(data=..., x="median_price", y="neighbourhood_cleansed", ax=ax)
        # OPTIONS :
        # order=... -> comme box plot, ou sans mais sort data before plotting
        # hue="col_name", palette={True: "#285e90", False: "lightgrey"}, legend=False 
            # -> colore les barres en fonction de col_name (True/False)
            # -> legend=False pour ne pas rajouter True/False en légende
    for container in ax.containers: # containers represent each group of bars drawn (containers[0]: below median ; containers[1]: above median)
        ax.bar_label(container, fmt="€%.0f", padding=3) # padding: gap, in points, between the end of a bar and its label
    ax.axvline(value, linestyle="--", color="grey") # -> vertical dashed line 

### SCATTER PLOTS 
    # General 
        sns.scatterplot(data=df, x="col_a", y="col_b", alpha=0.1, ax=ax)
            # avec alpha=0.1 permet de rendre un peu transparent les points et se rendre compte de la densité qui se cache derrière

    # Binned scatter: 
        # - cut x into equal-sized groups, then plot one point per group (the median of x against the median of y). 
        # - It shows the trend that a cloud of 40,000 points hides
        df["price_decile"] = pd.qcut(df["price_eur"], 10, labels=False)   # 0 to 9, about 10% of rows each
        Then use groupby("price_decile") with named aggregation, as usual

### SEVERAL PANELS IN ONE FIGURE 
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))   # 1 row, 2 columns
    sns.scatterplot(..., ax=axes[0])                    # left panel
    sns.lineplot(..., ax=axes[1])                       # right panel
    axes[0].set_title("...")                            # one title per panel
    fig.suptitle("...")                                 # one title for the whole figure

# Which chart to use:
    # line for change over time;
    # bar to compare categories;
    # histogram/box plot for a distribution;
        # histogram is better for one group in detail
        # box plot to compare a distibution across groups
    # scatter for the relationship between two numeric variables.



In [ ]:
### Task 1: the seasonality chart

import pandas as pd 
df_reviews = pd.read_csv("../data/raw/reviews.csv", parse_dates=["date"])
df_reviews["year"]=df_reviews["date"].dt.year
df_reviews["month"]=df_reviews["date"].dt.month
df_reviews=df_reviews.loc[(df_reviews["year"].between(2022,2025)) | ((df_reviews["year"]==2026) & (df_reviews["month"]<9)) ]

df_long = (
    df_reviews
    .groupby(["year", "month"], as_index=False)
    .agg(n_reviews = ("listing_id","count"))
)

import matplotlib.pyplot as plt
import seaborn as sns   
fig, ax = plt.subplots(figsize=(10, 5))                                 
sns.lineplot(data=df_long, x="month", y="n_reviews", hue="year", ax=ax) 
ax.set_title("June and October peaks, Jan.2022 - Aug.2026")
ax.set_xlabel("Month")
ax.set_ylabel("Number of reviews")
fig.text(0.01, -0.02,"Note: (i) data for September 2026 is partially missing so it is excluded (ii) data for October-December 2026 is entirely missing.", fontsize=9)
fig.savefig("../docs/fig_reviews_seasonality.png", dpi=150, bbox_inches="tight")

In [ ]:
### Task 2: compare the shapes, not the levels

import pandas as pd 
df_reviews = pd.read_csv("../data/raw/reviews.csv", parse_dates=["date"])
df_reviews["year"]=df_reviews["date"].dt.year
df_reviews["month"]=df_reviews["date"].dt.month
df_reviews=df_reviews.loc[(df_reviews["year"].between(2022,2025)) | ((df_reviews["year"]==2026) & (df_reviews["month"]<9)) ]

# Long table for charts
df_long = (
    df_reviews
    .groupby(["year", "month"], as_index=False)
    .agg(n_reviews = ("listing_id","count"))
)

# Keep complete years only 
df_long = df_long.loc[df_long.groupby("year")["month"].transform("count")==12]

# Month's share of its year's review
df_long["pct_of_year"]=(
    df_long["n_reviews"]
    / (df_long.groupby("year")["n_reviews"].transform("sum"))
).mul(100)

# Checks 
table_to_check = df_long.groupby("year", as_index=False)["pct_of_year"].sum()
display(table_to_check)
import numpy as np 
assert np.isclose(table_to_check["pct_of_year"],100,atol=0.15).all() 

# Charts 
import matplotlib.pyplot as plt
import seaborn as sns   
fig, ax = plt.subplots(figsize=(10, 5))                                 
sns.lineplot(data=df_long, x="month", y="pct_of_year", hue="year", ax=ax) 
ax.set_title("Reviews peak in June and October every year (2022–2025)")
ax.set_xlabel("Month")
ax.set_ylabel("Share of reviews (%)")
fig.text(0.01, -0.02,"Note: 2026 data is excluded because it is incomplete.", fontsize=9)
fig.savefig("../docs/fig_reviews_seasonality_shares.png", dpi=150, bbox_inches="tight")

# 25min

In [ ]:
### Task 3: the price distribution

import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns   
df = pd.read_parquet("../data/clean/listings_clean.parquet") 
df = df.loc[df["price_eur"].notna()]

# 1. Percentiles 
display(df["price_eur"].quantile([0.01, 0.25, 0.50, 0.75, 0.9, 0.99, 0.999, 1]))
p99=df["price_eur"].quantile(0.99)

# Prepare data for graphs 
df_no99=df.copy()
df_no99=df_no99.loc[df_no99["price_eur"]<=p99]
listings_p99=(len(df.loc[df["price_eur"]>p99]))

# 2. Histogram 
fig, ax = plt.subplots(figsize=(10,5))
sns.histplot(data=df_no99, x="price_eur", bins=50, ax=ax)
ax.set_title("Right skewed price distribution")
ax.set_xlabel("Price (€)")
ax.set_ylabel("Number of listings")
fig.text(0.00, -0.015, f"Note: listings with prices above the 99th percentile are excluded ({listings_p99} listings).")
fig.savefig("../docs/fig_price_histogram.png", dpi=150, bbox_inches="tight")

# 3. Box plot 

# Get price median order by room type 
order_by_median = (
    df_no99
    .groupby("room_type")["price_eur"]
    .median()
    .sort_values(ascending=False)
    .index
    .tolist()
)

fig, ax = plt.subplots(figsize=(10,5))
sns.boxplot(data=df_no99, x="price_eur", y="room_type", ax=ax, order=order_by_median)
ax.set_title("Least to most expensive room types")
ax.set_xlabel("Price (€)")
ax.set_ylabel("Room type")
fig.text(0.00, -0.015, f"Note: listings with prices above the 99th percentile are excluded ({listings_p99} listings).")
fig.savefig("../docs/fig_price_boxplot.png", dpi=150, bbox_inches="tight")

In [ ]:
### Task 4 : bar chart by neighbourhood

import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns   
df = pd.read_parquet("../data/clean/listings_clean.parquet") 
df = df.loc[(df["price_eur"].notna()) & (df["room_type"]=="Entire home/apt")]

median_entire_homes = df["price_eur"].median()
table = (
    df 
    .groupby("neighbourhood_cleansed")
    .agg(
        nb_listings = ("id","size"),
        median_price = ("price_eur", "median")
    )
    .sort_values("median_price", ascending=False)
)

fig, ax = plt.subplots(figsize=(10,5))
sns.barplot(data=table, x="median_price", y="neighbourhood_cleansed", ax=ax)
ax.axvline(median_entire_homes, linestyle="--", color="grey")
ax.set_title("Most (Élysée) to least (Ménilmontant) expensive neighbourhoods, entire homes only")
ax.set_xlabel("Median price (€)")
ax.set_ylabel("Neighbourhood")
ax.text(median_entire_homes+3,len(table)-1,"Paris-wide median for entire homes")
fig.savefig("../docs/fig_price_bar_chart.png", dpi=150, bbox_inches="tight")


In [ ]:
### Task 5: your first AI audit drill

# Using listings_clean.parquet, show whether bigger entire homes are cheaper per guest, and write a one-line conclusion.

df = pd.read_parquet("../data/clean/listings_clean.parquet")
homes = df[df["room_type"] == "Entire home/apt"]
homes["price_per_guest"] = homes["price_eur"] / homes["accommodates"]

summary = (
    homes.groupby("accommodates")
    .agg(
        price_per_guest_median = ("price_per_guest","median"),
        nb_listings = ("id", "size")
    )
    .reset_index()
)
display(summary)
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=summary, x="accommodates", y="price_per_guest", ax=ax)
ax.set_title("Larger homes are more expensive per guest")
ax.set_ylabel("Median price per guests (€)")
fig.savefig("../docs/fig_price_per_guest_median.png")

In [ ]:
### Task 6: are expensive homes booked less?

import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

# Import data 
df = pd.read_parquet("../data/clean/listings_clean.parquet")

# Prepare data 
df = df[df["room_type"]=="Entire home/apt"]
p99 = df["price_eur"].quantile(0.99)
listings_p99 = len(df[df["price_eur"]>p99])
df = df[df["price_eur"]<=p99]
assert df["price_eur"].notna().all()

df["price_decile"] = pd.qcut(df["price_eur"], 10, labels=False)
data_graph2 = (
    df 
    .groupby("price_decile")
    .agg(
        grouped_price_median = ("price_eur","median"),
        grouped_ltm_review_median = ("number_of_reviews_ltm", "median"),
        nb_listings = ("id", "size")
    )
)

# Graph 
fig, axes = plt.subplots(1, 2, figsize=(14, 5)) 
sns.scatterplot(data=df, x="price_eur", y="number_of_reviews_ltm", ax=axes[0], alpha=0.1)
sns.lineplot(data=data_graph2, x="grouped_price_median", y="grouped_ltm_review_median", ax=axes[1], marker="o")
axes[0].set_title("Scatter")
axes[0].set_xlabel("Price (€)")
axes[0].set_ylabel("Number of reviews (last 12 months)")
axes[1].set_title("Binned scatter")
axes[1].set_xlabel("Median price (€)")
axes[1].set_ylabel("Median number of reviews (last 12 months)")
fig.text(+0.07, -0.05, f"Note: entire homes with prices above the 99th percentile are excluded ({listings_p99} listings).")
fig.suptitle("Unclear whether more expensive entire homes get fewer reviews")
fig.savefig("../docs/fig_are_expensive_homes_booked_less.png", dpi=150, bbox_inches="tight")

# Sanity check 
assert data_graph2["nb_listings"].between(0.09*len(df), 0.11*len(df)).all()

# Issues with price_decile in the graph:
    # 1- decile 2, 5 and 8 are missing from the legend
    # 2- don't know how to rename "price_decile"

# Answer: it is unclear whether expensive homes are booked less: on one hand, the first figure display a decreasing trend between prices and number of reviews,
#         which would be in favor of the hypothesis, on the other hand, the second graph illustrates the opposite, with the median number of reviews going from 1 
#         for the first decile to 6 for the 5, 6, 7 and 8th deciles. The median number of review are at very low level (1 to 6) compared to the number of review 
#         a listing might have (many have more than 10 by looking at the first figure) because a lot of listings simply have zero reviews; to deal with those, 
#         we might want to reproduce the figures by excluding unreviewed listings, or perhaps taking the mean of number of reviews instead of the median.  
# Time : 1 hour. Got stuck with the sanity check + interpreting the result which show opposite thing

In [ ]:
### Task 7: a portfolio-quality chart

import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns   
df = pd.read_parquet("../data/clean/listings_clean.parquet") 
df = df.loc[(df["price_eur"].notna()) & (df["room_type"]=="Entire home/apt")]

median_entire_homes = df["price_eur"].median()
table = (
    df 
    .groupby("neighbourhood_cleansed")
    .agg(median_price = ("price_eur", "median"))
    .sort_values("median_price", ascending=False)
)
table["above_median"] = table["median_price"]>median_entire_homes

fig, ax = plt.subplots(figsize=(10,5))
sns.barplot(data=table, x="median_price", y="neighbourhood_cleansed", ax=ax, hue="above_median", palette={True: "#285e90", False: "lightgrey"}, legend=False) # hue to set different colours below and above median
sns.despine(bottom=True) # remove clutter
for container in ax.containers: # containers represent each group of bars drawn (containers[0]: below median ; containers[1]: above median)
    ax.bar_label(container, fmt="€%.0f", padding=3) # padding: gap, in points, between the end of a bar and its label
ax.axvline(median_entire_homes, linestyle="--", color="grey")
ax.set_xlabel("")
ax.set_xticks([])
ax.set_ylabel("")
fig.suptitle("Entire homes cost 2.3× more in Élysée than in Ménilmontant", x=0.01, ha="left", fontweight="bold")
ax.set_title("Median nightly price by neighbourhood; blue = above the Paris median", loc="left", fontsize=10)
ax.text(median_entire_homes+3,len(table)-1,f"Paris median: €{median_entire_homes:.0f}")
fig.text(-0.03, +0.02, f"Source: Inside Airbnb, Paris, snapshot of 16 Sep 2026. Priced entire homes, n = {len(df):,}.")
fig.savefig("../docs/fig_portfolio_price_by_neighbourhood.png", dpi=200, bbox_inches="tight")

# what did you mean by "loop over ax.containers"? Adding a loop for in the code?
# Issue 1: how to place neighbourhood names on the bars? median prices are with ax.bar_label(...), but I don't see the reference to the x-axis in it so I'm struggling finding the equivalent for the y-axis (also, what do containers and padding mean?).
# Issue 2: how to remove the median price numbers on the x-axis? As bars have their median written at their edge, numbers on the x-axis are noise.
